In [1]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, load_model
import glob
from Bio import SeqIO

In [2]:
# script for preparing MTB genomes for SD CNN predictions
genotype_dir = "/workspace/lab/mcarthurlab/wlodarsm/amr_phenotype_cnn/data/dummy_strain_fasta_files"

'''
LOCUS_ORDER = [
        "acpM-kasA",
        "gid",
        "rpsA",
        "clpC",
        "embCAB",
        "aftB-ubiA",
        "rrs-rrl",
        "ethAR",
        "oxyR-ahpC",
        "tlyA",
        "KatG",
        "rpsL",
        "rpoBC",
        "FabG1-inhA",
        "eis",
        "gyrBA",
        "panD",
        "pncA"
]
'''
DRUG = "CIP"

# AMI only: "rrs-rrl", "eis", 
LOCUS_ORDER = [
        "gyrBA"
]

BASE_TO_COLUMN = {'A': 0, 'C': 1, 'T': 2, 'G': 3, '-': 4}

In [3]:
def make_genotype_df(genotype_input_directory, locus_order=LOCUS_ORDER):
    """
	Parameters
	----------
	genotype_input_directory: str
		path to directory containing fasta files of genotype inputs

	Returns
	-------
	pd.DataFrame:
		indexed by isolate name, one column per locus
	"""
    # Make a df that combines all genotype data
    dfs_list = []

    for locus in locus_order:
        df_file = glob.glob(genotype_input_directory + "/" + locus + "*.fasta")[0]
        print("reading fasta file", df_file)
        _df = sequence_dictionary(df_file)
        dfs_list.append(_df)

    df_genos = dfs_list[0].join(dfs_list[1:], how='outer')
    return df_genos

In [4]:
def sequence_dictionary(filename):
    """
	Creates a dataframe that contains the sequence of each locus for each isolate
	Note that this function splits the identifier names in the fasta file on '/'
	and takes the last entry

	Parameters
	----------
	filename: str
		path to directory containing genotype data (one fasta file containing
		sequences for all isolates at a particular locus)

	Returns
	-------
	pd.DataFrame with one column, indexed by strain name
		column name will be the beginning string of the file name
	"""
    seq_dict = SeqIO.to_dict(
        SeqIO.parse(filename, "fasta"),
        key_function=lambda x: x.id.split("/")[-1].split(".cut")[0])

    # create a dictionary of identifier: sequence
    for identifier, sequence in seq_dict.items():
        seq_dict[identifier] = str(sequence.seq)

    df = pd.DataFrame.from_dict(seq_dict, orient='index')
    gene_name = filename.split("/")[-1].split("_")[0]
    df.columns = [gene_name]

    return df

In [5]:
def get_one_hot(sequence):
    """
	Creates a one-hot encoding of a sequence
	Parameters
	----------
	sequence: iterable of str
		Sequence containing only ACTG- characters

	Returns
	-------
	np.ndarray of int
		L (seq len) x 5 one-hot encoded sequence
	"""

    seq_len = len(sequence)
    seq_in_index = [BASE_TO_COLUMN.get(b, b) for b in sequence]
    one_hot = np.zeros((seq_len, 5))

    # Assign the found positions to 1
    one_hot[np.arange(seq_len), np.array(seq_in_index)] = 1

    return one_hot

In [6]:
def create_X(df_geno_pheno):
    """
	Create an input X matrix, with output dimensions:
		n_strains x 5 (one-hot) x longest locus length x no. of loci

    Parameters
    ----------
    df_geno_pheno: pd.DataFrame
        generated by make_geno_pheno_pkl, contains the numeric encoded
        phenotype information and the one-hot encoded sequence information for each isolate

    Returns
    -------
    np.ndarray
        with shape N_strains, 5, L_longest_locus, N_loci
        contains the one-hot encoded sequence information for each locus for each strain
	"""

    def _get_shapes(df_geno_pheno):
        """
		Finds the length of each gene in the input dataframe
		Parameters
		----------
		df_geno_pheno: pd.Dataframe

		Returns
		-------
		dict of str: int
			length of coordinates in each column
		"""
        shapes = {}
        for column in df_geno_pheno.columns:
            if "one_hot" in column:
                shapes[column] = df_geno_pheno.loc[df_geno_pheno.index[0], column].shape[0]

        return shapes

    shapes = _get_shapes(df_geno_pheno)

    # Length of longest gene locus
    n_genes = len(shapes)
    L_longest = max(list(shapes.values()))
    print("found n genes", n_genes, "and longest gene", L_longest)

    # Number of strains in model
    n_strains = df_geno_pheno.shape[0]

    # define shape of matrix - fill with zeros (effectively accomplishes padding)
    X = np.zeros((n_strains, 5, L_longest, n_genes))

    # for each strain and gene locus
    for idx, strain in enumerate(df_geno_pheno.index):
        for gene_index, gene in enumerate(shapes.keys()):
            one_hot_gene = df_geno_pheno.loc[strain, gene]
            X[idx, :, range(0, one_hot_gene.shape[0]), gene_index] = one_hot_gene

    return X

In [7]:
df_genos = make_genotype_df(genotype_dir)
df_genos = df_genos.dropna(axis="index")
df_genos.columns = [c.replace(".fasta", "") for c in df_genos.columns]
df_genos

reading fasta file /workspace/lab/mcarthurlab/wlodarsm/amr_phenotype_cnn/data/dummy_strain_fasta_files/gyrBA.fasta


,gyrBA
h37rv,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...
h37rv_bINH,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...
h37rv_bINH_G_5074.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...
h37rv_bINH_-_5074.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...
h37rv_bINH_-_5075.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...
...,...
h37rv_bINH_-_4408309.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...
h37rv_bINH_C_4408309.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...
h37rv_bINH_G_4408309.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...
h37rv_bINH_-_4408315.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...


In [8]:
#apply one hot encoding to each column
for column in df_genos.columns:
    df_genos[column + "_one_hot"] = df_genos[column].apply(get_one_hot)

In [9]:
df_genos.head()

,gyrBA,gyrBA_one_hot
h37rv,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...,"[[0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0...."
h37rv_bINH,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...,"[[0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0...."
h37rv_bINH_G_5074.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...,"[[0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0...."
h37rv_bINH_-_5074.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...,"[[0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0...."
h37rv_bINH_-_5075.0,CACGTCGATCGGCCCAGAACAAGGCGCTCCGGTCCCGGCCTGAGAG...,"[[0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0...."


In [10]:
#create input matrix X
X = create_X(df_genos)
print(X.shape)

found n genes 1 and longest gene 4859


(9202, 5, 4859, 1)


In [11]:
np.save(f"/workspace/lab/mcarthurlab/wlodarsm/amr_phenotype_cnn/data/sd_matrices/X_{DRUG}.npy", X)
np.save(f"/workspace/lab/mcarthurlab/wlodarsm/amr_phenotype_cnn/data/sd_matrices/strain_ids_{DRUG}.npy", df_genos.index.values)

In [12]:
X = np.load(f"/workspace/lab/mcarthurlab/wlodarsm/amr_phenotype_cnn/data/sd_matrices/X_{DRUG}.npy")
strain_ids = np.load(f"/workspace/lab/mcarthurlab/wlodarsm/amr_phenotype_cnn/data/sd_matrices/strain_ids_{DRUG}.npy", allow_pickle=True)

In [13]:
model = tf.keras.models.load_model(
    f"/workspace/lab/mcarthurlab/wlodarsm/amr_phenotype_cnn/models/paper_sd_models/{DRUG}_tf1_model",
    compile=False
)

model.summary()

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
If using Keras pass *_constraint arguments to layers.
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 1, 4848, 64)       3904      
_________________________________________________________________
conv1d (Conv2D)              (None, 1, 4837, 64)       49216     
_________________________________________________________________
max_pooling1d (MaxPooling2D) (None, 1, 1612, 64)       0         
_________________________________________________________________
conv1d_1 (Conv2D)            (None, 1, 1610, 32)       6176      
_________________________________________________________________
con

2026-03-26 11:31:53.634770: I tensorflow/core/platform/cpu_feature_guard.cc:142] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 AVX512F FMA
2026-03-26 11:31:53.656906: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 3100335000 Hz
2026-03-26 11:31:53.675553: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x92b5c470 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-03-26 11:31:53.675649: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version


In [14]:
preds = model.predict(X).flatten()


In [15]:
print("min:", preds.min())
print("max:", preds.max())
print("mean:", preds.mean())
print("std:", preds.std())

min: 0.009643495
max: 0.910567
mean: 0.6285888
std: 0.019941434
